# Angkor · The city that moved water

An interactive engineering atlas of the historical water system, built with **native SVG, CSS and JavaScript**. Run all cells to generate `output/20261031_angkor_hydraulic_city.html`. No network requests, map tiles or JavaScript packages are needed. The previous Angkor notebook is retained separately.

## Reading the reconstruction

This is a **cumulative, interpretive schematic of the 9th–13th centuries**, not a reconstruction of one year. Relative placement is simplified: uplands north, West Baray west of Angkor Thom, East Baray east, Angkor Wat south of Angkor Thom, Tonle Sap farther south. Distances, canal alignments, field parcels, lake shoreline and elevations are not surveyed geometry. Arrows express broad functional relationships, not verified links between every drawn node. Reservoirs did not necessarily all operate together.

The seasonal views are qualitative explanations. Animated dashes indicate direction only; neither speed, line width nor color represents discharge. No historical water levels, yields or irrigation capacities are inferred. The small reservoir comparison alone uses a common metric scale: approximate plan area = published length × width, **not** usable water surface or storage volume.

All data, diagram geometry, page layout, controls and callbacks live in this notebook. Shared branding and the legal footer come from `site_export.py`.

## Evidence and editorial choices

1. Fletcher et al. (2008), [The water management network of Angkor, Cambodia](https://doi.org/10.1017/S0003598X00097295), *Antiquity* 82, 658–670. Basis for the three functional zones, northern collection, central retention and southern dispersal; West Baray approximately 8 × 2 km. [Open paper](https://angkordatabase.asia/libs/docs/publications/water-management-in-angkor/The_water_management_network_of_Angkor_C.pdf).
2. Kummu (2009), [Water management in Angkor: Human impacts on hydrology and sediment transportation](https://pubmed.ncbi.nlm.nih.gov/18926615/), *Journal of Environmental Management* 90, 1413–1421. Network exceeding 1,000 km², monsoon seasonality, changed catchments and erosion/sedimentation.
3. Penny et al. (2018), [The demise of Angkor: Systemic vulnerability of urban infrastructure to climatic variations](https://pmc.ncbi.nlm.nih.gov/articles/PMC6192684/), *Science Advances* 4, eaau4029. Multiple roles, long development, late-12th-century Jayatataka and modeled vulnerability to extreme flows. Our illustration does not reproduce their numerical model.
4. ICCROM (2020), [Traditional Knowledge Systems and the Conservation and Management of Asia’s Heritage](https://www.iccrom.org/sites/default/files/publications/2020-10/traditional-knowledge-systems.pdf), chapter 6, p. 90. East Baray approximately 7.5 × 1.8 km and late-9th/early-10th-century construction. Published dimensions vary; rounded values are used consistently here.

The northern catchment, smaller ponds, fields and connecting routes below are original schematic drawings. Their density is illustrative. Religious and hydraulic functions coexisted; this atlas does not reduce the temples to utilitarian infrastructure. Historical uncertainty is surfaced in the page as well as here.

In [8]:
from pathlib import Path
import html
import json
import sys
from IPython.display import IFrame, display

current = Path.cwd().resolve()
project_root = next((p for p in (current, *current.parents)
                     if (p / 'site_export.py').is_file() and (p / 'notebooks').is_dir()), None)
if project_root is None:
    raise RuntimeError('Run from the repository or notebooks directory.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from site_export import write_svg_page
output_file = project_root / 'output' / '20261031_angkor_hydraulic_city.html'

sources = {
    'fletcher': {'label': 'Fletcher et al. · 2008', 'url': 'https://doi.org/10.1017/S0003598X00097295'},
    'kummu': {'label': 'Kummu · 2009', 'url': 'https://pubmed.ncbi.nlm.nih.gov/18926615/'},
    'penny': {'label': 'Penny et al. · 2018', 'url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC6192684/'},
    'iccrom': {'label': 'ICCROM · 2020, p. 90', 'url': 'https://www.iccrom.org/sites/default/files/publications/2020-10/traditional-knowledge-systems.pdf'},
}
reservoirs = [
    {'id': 'west', 'name': 'West Baray', 'length_km': 8.0, 'width_km': 2.0, 'source': 'fletcher'},
    {'id': 'east', 'name': 'East Baray', 'length_km': 7.5, 'width_km': 1.8, 'source': 'iccrom'},
]
for reservoir in reservoirs:
    reservoir['plan_area_km2'] = reservoir['length_km'] * reservoir['width_km']
assert [r['plan_area_km2'] for r in reservoirs] == [16.0, 13.5]
reservoirs

[{'id': 'west',
  'name': 'West Baray',
  'length_km': 8.0,
  'width_km': 2.0,
  'source': 'fletcher',
  'plan_area_km2': 16.0},
 {'id': 'east',
  'name': 'East Baray',
  'length_km': 7.5,
  'width_km': 1.8,
  'source': 'iccrom',
  'plan_area_km2': 13.5}]

## The interpretive model

The atlas has three independent states: **season**, **journey stage**, and **selected feature**. Feature selection explains a component; journey selection emphasizes its region. Wet/dry views change visual emphasis and explanatory text, never a numeric historical estimate. Reservoir outlines remain fixed: reduced dry-season emphasis is not a mapped shoreline.

In [9]:
features = {
 'catchment': {'name': 'Northern catchments', 'kind': '01 / COLLECT', 'metric': 'Monsoon → channels', 'body': 'Seasonal rain and upland runoff enter a landscape reshaped by canals and embankments. The challenge is managing both abundance and scarcity.', 'detail': 'Changing river courses also changed catchments. Sediment and erosion made maintenance a continuing task.', 'source': 'kummu', 'stage': 1},
 'west': {'name': 'West Baray', 'kind': '02 / STORE', 'metric': '8 × 2 km', 'body': 'An immense embanked reservoir in the central retention zone. Its rectangular footprint makes the scale of the engineering visible.', 'detail': 'Approximate plan area: 16 km². This is a geometric footprint, not a storage-volume estimate.', 'source': 'fletcher', 'stage': 2},
 'east': {'name': 'East Baray', 'kind': '02 / STORE', 'metric': '7.5 × 1.8 km', 'body': 'An earlier great reservoir, established in the late ninth to early tenth century. Successive rulers reshaped the water landscape over generations.', 'detail': 'Approximate plan area: 13.5 km². Dimensions are rounded; published measurements differ.', 'source': 'iccrom', 'stage': 2},
 'jay': {'name': 'Jayatataka', 'kind': '02 / STORE', 'metric': 'Late 12th century', 'body': 'The last of the major barays was inserted into an already extensive network. Angkor’s infrastructure evolved with its urban form.', 'detail': 'Its inclusion shows accumulated infrastructure, not proof that every reservoir operated simultaneously.', 'source': 'penny', 'stage': 2},
 'thom': {'name': 'Angkor Thom', 'kind': '02 / RETAIN', 'metric': 'A city within the network', 'body': 'The city enclosure and its moat sit among reservoirs and channels. Sacred space, settlement and water management were intertwined.', 'detail': 'The enclosure is simplified. Its symbolic geometry and hydraulic role belong to the same landscape.', 'source': 'fletcher', 'stage': 2},
 'wat': {'name': 'Angkor Wat', 'kind': '02 / RETAIN', 'metric': 'Temple + moat', 'body': 'A monumental enclosure within a much larger water landscape. Moats were among the network’s central retention features.', 'detail': 'The diagram includes a southern disposal relationship, not a detailed reconstruction of the temple’s plumbing.', 'source': 'fletcher', 'stage': 2},
 'fields': {'name': 'The inhabited plain', 'kind': '03 / DISTRIBUTE', 'metric': 'Water across a landscape', 'body': 'Canals, local ponds and agricultural land formed a dispersed urban landscape. Water management supported agriculture and moderated flooding.', 'detail': 'The drawn fields and ponds indicate a landscape type. They are not mapped parcels or an estimate of irrigated area.', 'source': 'penny', 'stage': 3},
 'lake': {'name': 'Toward Tonle Sap', 'kind': '03 / DRAIN', 'metric': 'A route for surplus', 'body': 'Southern channels carried water away from the central zone. Disposal was as essential as retention.', 'detail': 'Regional drainage is shown toward the lake. This does not depict Tonle Sap’s separate seasonal exchange with the Mekong.', 'source': 'fletcher', 'stage': 3},
}
stages = [
 {'name': 'The whole system', 'heading': 'An engineered landscape', 'text': 'Follow water from the northern catchments through the reservoirs and inhabited plain to the lake. Select a numbered stage or a named feature.', 'feature': None},
 {'name': 'Collect', 'heading': 'Catch it. Slow it. Redirect it.', 'text': 'The northern zone gathers and redirects runoff. The drawing highlights the broad journey from uplands toward the central reservoirs.', 'feature': 'catchment'},
 {'name': 'Store', 'heading': 'Make the monsoon last.', 'text': 'Central reservoirs and moats retain water. Their relationship to the wider network matters as much as the size of any single basin.', 'feature': 'west'},
 {'name': 'Distribute & drain', 'heading': 'Give water a way through.', 'text': 'Across the southern plain, distribution and drainage work together. Managing surplus helps protect the city as well as sustain its landscape.', 'feature': 'fields'},
]
seasons = {
 'wet': {'label': 'Monsoon', 'heading': 'When water is abundant', 'text': 'Incoming runoff, retention and surplus drainage are emphasized. The system has to accommodate water as well as keep it.'},
 'dry': {'label': 'Dry season', 'heading': 'When water is scarce', 'text': 'Reservoirs and local water stores take visual priority. Less incoming water makes seasonal retention valuable; actual historical levels are unknown.'},
}
model = {'features': features, 'stages': stages, 'seasons': seasons, 'sources': sources}

## Diagram geometry

Coordinates below are drawing units, not latitude/longitude. All canal paths are ordered upstream → downstream so moving dashes and static arrows agree. Larger fixed arrowheads are also placed along each route, keeping direction readable when motion is paused. The regional supply is northern rainfall/runoff; surplus drains south toward Tonle Sap (Fletcher et al., 2008, pp. 658, 663). Reservoir and temple shapes are selectable with pointer, touch, Enter or Space. The two plan-area comparison rectangles use **35 drawing units per kilometre** on both axes.

In [10]:
def esc(value):
    return html.escape(str(value), quote=True)

def label(x, y, text, cls='map-label', anchor='start'):
    return f'<text x="{x}" y="{y}" class="{cls}" text-anchor="{anchor}">{esc(text)}</text>'

def selectable(key, drawing):
    return (f'<g class="feature" data-feature="{key}" data-zone="{features[key]["stage"]}" '
            f'role="button" tabindex="0" aria-label="Explore {esc(features[key]["name"])}" aria-pressed="false">'
            f'<title>{esc(features[key]["name"])}</title>{drawing}</g>')

parts = ["""<svg id="network" viewBox="0 0 1000 730" xmlns="http://www.w3.org/2000/svg" aria-labelledby="map-title map-description">
<title id="map-title">Angkor’s historical water system: a functional schematic</title>
<desc id="map-description">North is at the top. Water moves from uplands through central reservoirs and moats, then across the southern plain toward Tonle Sap. Select a named feature for details. Canal paths and field boundaries are illustrative.</desc>
<defs>
 <pattern id="grid" width="40" height="40" patternUnits="userSpaceOnUse"><path d="M40 0H0V40" fill="none" stroke="#8ba7a0" stroke-opacity=".055"/></pattern>
 <pattern id="water-lines" width="12" height="8" patternUnits="userSpaceOnUse"><path d="M0 4H12" stroke="#82d7d3" stroke-opacity=".17"/></pattern>
 <pattern id="field-lines" width="27" height="22" patternUnits="userSpaceOnUse" patternTransform="rotate(-7)"><rect x="2" y="2" width="23" height="18" fill="#60794c" fill-opacity=".2" stroke="#a6bb73" stroke-opacity=".24" stroke-width=".7"/></pattern>
 <linearGradient id="lake-fill" x2="0" y2="1"><stop stop-color="#206b72" stop-opacity=".35"/><stop offset="1" stop-color="#163e4d" stop-opacity=".1"/></linearGradient>
 <linearGradient id="reservoir-fill" x2="0" y2="1"><stop stop-color="#1c6870"/><stop offset="1" stop-color="#163f4a"/></linearGradient>
 <marker id="arrow" viewBox="0 0 10 10" refX="7" refY="5" markerWidth="8" markerHeight="8" orient="auto"><path d="M1 1L8 5L1 9" fill="none" stroke="#8ce1dc" stroke-width="1.3"/></marker>
</defs>
<rect width="1000" height="730" fill="url(#grid)"/>
"""]
# Stylized uplands: original vector contours rather than modern terrain tiles.
for i in range(10):
    y = 18+i*12
    parts.append(f'<path class="contour" d="M180 {y+25} C250 {y-60} 305 {y+50} 380 {y+3} S485 {y+35} 555 {y-8} S685 {y+35} 765 {y-15} S855 {y+35} 945 {y-12}"/>')
parts.append(label(43, 45, 'N', 'north'))
parts.append('<path d="M49 94V57M43 68L49 57L55 68" stroke="#c6cabb" fill="none"/>')
parts.append(label(84, 83, 'UPLANDS', 'zone-label'))
parts.append(label(84, 100, 'Northern catchments', 'small-label'))
parts.append(label(21, 230, '01', 'zone-number'))
parts.append(label(21, 251, 'COLLECT', 'zone-label'))
parts.append(label(21, 375, '02', 'zone-number'))
parts.append(label(21, 396, 'STORE', 'zone-label'))
parts.append(label(21, 553, '03', 'zone-number'))
parts.append(label(21, 574, 'DISTRIBUTE', 'zone-label'))
parts.append(label(21, 591, '& DRAIN', 'zone-label'))
for y in [278,454]:
    parts.append(f'<path d="M20 {y}H970" stroke="#bdc8b4" stroke-opacity=".13" stroke-dasharray="2 9"/>')
parts.append(selectable('catchment', '<rect class="hit" x="350" y="55" width="440" height="90" rx="8"/>' + label(565,89,'KULEN HILLS & NORTHERN WATERSHEDS','upland-title','middle') + label(565,112,'WATER INPUT • rain & northern runoff','small-label','middle')))
# Farmland is a symbol of dispersed agricultural settlement, not a mapped survey.
parts.append(selectable('fields', '<path class="fields" d="M154 474L391 464L415 609L184 625Z M527 473L832 455L891 588L565 635Z" fill="url(#field-lines)"/>' + '<rect class="hit" x="623" y="533" width="205" height="52" rx="4"/>' + label(725,555,'FIELDS & LOCAL PONDS','field-title','middle') + label(725,574,'The inhabited plain','small-label','middle')))
# An abstract wetland/lake edge, not today’s shoreline.
parts.append(selectable('lake', '<path class="lake" d="M135 666Q200 627 267 655T410 648T563 654T725 636T973 663V730H135Z" fill="url(#lake-fill)" stroke="#4b9da3" stroke-opacity=".5"/>' + label(555,692,'TONLE SAP','lake-label','middle') + label(555,713,'RECEIVES SURPLUS • drainage from Angkor','small-label','middle')))

# Broad conceptual connections. No invented canal names or precision claims.
routes = [
 (1,'inflow','M380 142V175H315V213H370V294'),
 (1,'inflow','M548 141V179H515V230H490V298'),
 (1,'inflow','M713 140V179H801V232H862V294'),
 (1,'inflow','M602 142V198H668V229'),
 (1,'inflow','M288 169V225H181V294'),
 (1,'bypass','M315 213H145V420L194 641'),
 (1,'bypass','M801 232H928V458L881 642'),
 (2,'transfer','M562 255H535V298'),
 (2,'transfer','M668 270V283H720V294'),
 (2,'transfer','M405 334H448'),
 (2,'transfer','M573 334H622'),
 (2,'release','M210 365V437H279V481'),
 (2,'release','M880 330H910V437H828V483'),
 (2,'release','M511 393V422'),
 (3,'distribution','M279 481V605L255 650'),
 (3,'distribution','M279 481L427 499H645L828 483'),
 (3,'distribution','M512 492V591L538 651'),
 (3,'distribution','M828 483V605L793 643'),
 (3,'distribution','M279 540H362V612'),
 (3,'distribution','M645 499V604H701V641'),
 (3,'distribution','M194 511H279'),
 (3,'distribution','M362 555H426V606'),
]
for zone, kind, d in routes:
    parts.append(f'<g class="route {kind}" data-zone="{zone}"><path class="channel-bank" d="{d}"/><path class="channel" d="{d}"/><path class="flow" d="{d}"/><path class="direction" d="{d}" marker-end="url(#arrow)"/></g>')
# Local ponds: illustrative, deliberately uncounted in the narrative.
for x,y in [(216,500),(238,572),(323,518),(387,582),(588,524),(609,579),(757,603),(843,518)]:
    parts.append(f'<rect class="pond" data-zone="3" x="{x}" y="{y}" width="13" height="9"/>')

def baray(key,x,y,w,h,name,sub):
    return selectable(key, f'<rect class="selection-ring" x="{x-6}" y="{y-6}" width="{w+12}" height="{h+12}" rx="3"/><rect class="bank" x="{x-3}" y="{y-3}" width="{w+6}" height="{h+6}"/><rect class="basin" x="{x}" y="{y}" width="{w}" height="{h}"/><rect x="{x}" y="{y}" width="{w}" height="{h}" fill="url(#water-lines)"/>' + label(x+w/2,y+h/2-2,name,'reservoir-label','middle') + label(x+w/2,y+h/2+18,sub,'reservoir-sub','middle'))
parts.append(baray('west',165,297,240,68,'WEST BARAY','11th century'))
parts.append(baray('east',622,297,258,65,'EAST BARAY','Late 9th–early 10th century'))
parts.append(baray('jay',562,232,130,38,'JAYATATAKA','Late 12th century'))

def temple(key,x,y,w,h,name):
    cx, cy = x+w/2, y+h/2
    drawing = f'<rect class="selection-ring" x="{x-8}" y="{y-8}" width="{w+16}" height="{h+16}" rx="4"/><rect class="moat" x="{x}" y="{y}" width="{w}" height="{h}"/><rect class="enclosure" x="{x+9}" y="{y+9}" width="{w-18}" height="{h-18}"/><rect class="temple-inner" x="{cx-16}" y="{cy-13}" width="32" height="26"/><path class="temple-lines" d="M{cx-24} {cy}H{cx+24}M{cx} {cy-22}V{cy+22}"/>'
    for dx,dy in [(0,0),(-9,-6),(9,-6),(-9,6),(9,6)]:
        drawing += f'<rect class="tower" x="{cx+dx-2}" y="{cy+dy-2}" width="4" height="4"/>'
    return selectable(key,drawing+label(cx,y+h+20,name,'temple-label','middle'))
parts.append(temple('thom',449,298,124,95,'ANGKOR THOM'))
parts.append(temple('wat',476,432,72,55,'ANGKOR WAT'))
parts.append(label(318,197,'Northern collection','route-label'))
parts.append(label(907,405,'Surplus','small-label','end'))
parts.append(label(907,422,'bypass','small-label','end'))
parts.append(label(20,710,'SCHEMATIC / NOT TO SCALE','zone-label'))
parts.append('</svg>')
network_svg = ''.join(parts)

comparison = ['<svg viewBox="0 0 590 235" role="img" aria-labelledby="area-title"><title id="area-title">Reservoir footprints at a common scale: West Baray approximately 16 square kilometres; East Baray approximately 13.5 square kilometres.</title>']
for r,y in zip(reservoirs,[34,139]):
    w,h = r['length_km']*35,r['width_km']*35
    comparison += [f'<rect x="1" y="{y}" width="{w}" height="{h}" class="area-box"/>', label(14,y+25,r['name'].upper(),'reservoir-label'), label(14,y+45,f"{r['length_km']:g} × {r['width_km']:g} km",'reservoir-sub'), label(320,y+29,f"≈ {r['plan_area_km2']:g} km²",'area-value'), label(320,y+49,'approximate plan area','small-label')]
comparison += ['<path d="M1 225H71M1 220V230M71 220V230" stroke="#98a8a4"/>',label(82,229,'2 km • common scale','small-label'),'</svg>']
comparison_svg = ''.join(comparison)

## Page styling and responsive layout

The main atlas keeps its aspect ratio. On narrow screens it becomes horizontally scrollable with explicit zoom controls; the explanatory panel follows beneath it. Motion can be paused, honors reduced-motion preferences and pauses when the tab is hidden. SVG text remains selectable, and every interactive feature has a keyboard equivalent in the inspector's selector.

In [11]:
css = r"""
:root{color-scheme:dark;--bg:#101e24;--panel:#14262d;--line:#304047;--muted:#9daea9;--ink:#eef0df;--water:#7bd6d0;--gold:#dcb77b;--green:#b3c68c}
*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--ink);font:15px/1.6 'Segoe UI',Arial,sans-serif}a{color:var(--water);text-underline-offset:4px}button,select{font:inherit}button{cursor:pointer}button:focus-visible,select:focus-visible,summary:focus-visible{outline:2px solid var(--gold);outline-offset:4px}button{color:var(--muted);background:transparent;border:1px solid var(--line);border-radius:3px;padding:9px 14px;min-height:44px}button:hover{color:var(--ink);border-color:var(--water)}button[aria-pressed=true]{background:#284348;color:#c6f3e8;border-color:#609b9b}
/* Keep the home-link logo compact when shared CSS is unavailable in a local preview. */
.site-brand{display:inline-flex;align-items:center;gap:.65rem;margin:24px 4vw 0!important;color:#c7d0dd;font:1.15rem/1 system-ui,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;text-decoration:none}
.site-brand svg{width:28px;height:31px;flex:none;color:#a9bbd2}
.site-brand-copy{display:grid;gap:.35rem}
.site-brand-name{letter-spacing:-.045em}
.site-brand-tagline{font-size:.48rem;letter-spacing:.19em;text-transform:uppercase}
.site-brand:hover{color:#fff}
.site-footer{background:var(--bg)!important;border-color:var(--line)!important}main{max-width:1460px;margin:auto;padding:26px 4vw 60px}.eyebrow{text-transform:uppercase;letter-spacing:.19em;font-size:11px;font-weight:600;color:var(--gold);margin:0 0 18px}.hero{display:grid;grid-template-columns:1fr 400px;gap:40px;align-items:end;padding:22px 0 32px;border-bottom:1px solid var(--line)}h1{font:normal clamp(64px,8.5vw,118px)/.85 Georgia,serif;letter-spacing:-.045em;margin:0 0 20px}h1 span{display:block;font:normal clamp(23px,2.7vw,37px)/1.25 Georgia,serif;letter-spacing:-.025em;margin-top:20px}.intro{font-size:16px;color:#b9c6bf;margin:0 0 20px;max-width:43ch}.hero-meta{display:flex;gap:28px;padding-top:13px;border-top:1px solid var(--line)}.hero-meta strong{display:block;color:var(--ink);font:normal 25px Georgia,serif}.hero-meta span{color:var(--muted);font-size:11px;letter-spacing:.05em}.toolbar{display:flex;align-items:center;justify-content:space-between;gap:16px;margin:22px 0 16px}.tool-group{display:flex;gap:5px;align-items:center;flex-wrap:wrap}.tool-label{font-size:10px;letter-spacing:.15em;margin-right:12px;color:var(--muted)}.small-button{font-size:12px}.atlas{border:1px solid var(--line);background:radial-gradient(ellipse at 35% 20%,#1a3033,transparent 70%),var(--bg);display:grid;grid-template-columns:minmax(0,1fr) 285px}.map-panel{min-width:0;position:relative}.map-top{padding:17px 20px 4px;display:flex;align-items:center;justify-content:space-between;color:var(--muted);font-size:11px;gap:12px}.live-dot{display:inline-block;width:6px;height:6px;background:var(--water);border-radius:50%;margin-right:8px}.map-scroll{overflow-x:auto;scrollbar-color:#648d8e var(--bg)}#network{display:block;width:100%;min-width:600px;height:auto}#network.zoomed{width:145%;min-width:850px}.map-hint{margin:0;padding:4px 20px 12px;font-size:11px;color:var(--muted)}.legend{display:flex;gap:17px;flex-wrap:wrap;padding:12px 20px;border-top:1px solid var(--line);font-size:10px;letter-spacing:.03em;color:#b4c2ba}.legend i{display:inline-block;width:18px;margin-right:6px;vertical-align:middle}.legend .water-key{border-top:2px solid var(--water)}.legend .gold-key{height:9px;border:1px solid var(--gold)}.legend .green-key{height:9px;background:#4e6343}.legend .arrow-key{color:var(--water);font-style:normal;font-size:16px}.inspector{padding:24px 23px;background:#14262ddd;border-left:1px solid var(--line);display:flex;flex-direction:column}.inspector label{font-size:10px;letter-spacing:.15em;color:var(--muted)}select{width:100%;background:#102127;color:var(--ink);border:1px solid #526663;padding:10px 9px;border-radius:3px;margin:8px 0 27px;min-height:44px}.inspect-kind{font-size:10px;letter-spacing:.16em;color:var(--gold);margin:0 0 11px}.inspector h2{font:normal 30px/1.2 Georgia,serif;margin:0 0 18px}.metric{font:normal 22px/1.3 Georgia,serif;color:var(--water);margin:0 0 16px}.inspector p{font-size:13px;line-height:1.75}.inspect-body{color:#d0d9ce;margin:0 0 14px}.inspect-detail{color:var(--muted);border-top:1px solid var(--line);padding-top:14px}.source-link{font-size:11px}.season-note{margin-top:auto;padding-top:23px}.season-note h3{font-size:11px;font-weight:600;color:var(--water);margin:0}.season-note p{font-size:12px;color:var(--muted);margin:7px 0 0}.journey{display:grid;grid-template-columns:repeat(4,1fr);margin-top:14px;border:1px solid var(--line)}.journey button{border:0;border-right:1px solid var(--line);border-radius:0;text-align:left;padding:15px 18px}.journey button:last-child{border-right:0}.journey small{display:block;font-size:10px;letter-spacing:.12em;margin-bottom:5px}.journey span{font:19px Georgia,serif}.story{display:grid;grid-template-columns:1fr 1.3fr;gap:36px;margin:22px 0 42px;align-items:start}.story h2{font:normal 25px/1.2 Georgia,serif;margin:0}.story p{font-size:13px;color:var(--muted);margin:0}.secondary{display:grid;grid-template-columns:1fr 1fr;gap:55px;border-top:1px solid var(--line);padding-top:28px}.secondary h2{font:normal 27px Georgia,serif;margin:0 0 12px}.secondary .eyebrow{margin-bottom:13px}.secondary p{font-size:13px;color:var(--muted);max-width:60ch}.secondary svg{width:100%;height:auto}.section-label{font:12px 'Segoe UI',sans-serif;fill:#c6d0c5}.section-water{stroke:var(--water);fill:none;stroke-width:2}.section-ground{fill:#514836;fill-opacity:.4;stroke:#a89470;stroke-width:1}.caption{font-size:11px!important;color:var(--muted);line-height:1.7}.evidence{margin-top:35px;border-top:1px solid var(--line);border-bottom:1px solid var(--line);padding:18px 0}.evidence summary{cursor:pointer;list-style-position:inside;color:var(--gold);font-size:13px}.evidence-grid{display:grid;grid-template-columns:1fr 1fr;gap:45px;padding:18px 0 12px}.evidence p,.evidence li{font-size:12px;color:#b4c4ba}.evidence h3{font:normal 20px Georgia,serif}.evidence ul{padding-left:18px}.page-note{margin-top:18px;font-size:11px;color:var(--muted)}
/* Original SVG marks */
.contour{stroke:#b29d69;stroke-opacity:.15;stroke-width:.8;fill:none}.map-label{fill:var(--ink)}.north{fill:#d1d7c8;font:15px Georgia,serif}.zone-number{fill:#bac9bd;opacity:.45;font:33px Georgia,serif}.zone-label{fill:#9caea4;font:9px 'Segoe UI',sans-serif;letter-spacing:1.4px}.small-label{fill:#a9bcb3;font:11px 'Segoe UI',sans-serif}.upland-title{fill:#c6cdb8;font:12px 'Segoe UI',sans-serif;letter-spacing:1.4px}.field-title{fill:#c2ce9d;font:11px 'Segoe UI',sans-serif;letter-spacing:1px}.route-label{fill:#93b8b2;font:10px 'Segoe UI',sans-serif}.channel-bank{stroke:#69918a;stroke-opacity:.16;stroke-width:9;fill:none}.channel{stroke:#458c90;stroke-width:2.6;fill:none}.flow{stroke:#9bebe1;stroke-width:2;fill:none;stroke-dasharray:3 24;animation:flow 3s linear infinite;opacity:.85}.direction{fill:none;stroke:transparent;stroke-width:1}.flow-arrow{fill:none;stroke:#b8f4e7;stroke-width:2;stroke-linecap:round;stroke-linejoin:round;pointer-events:none}.flow-guide{margin:8px 20px 12px;padding:10px 12px;border-left:2px solid var(--water);background:#1d353a;color:#c6ded4;font-size:12px;line-height:1.6}.flow-guide strong{color:var(--water);font-weight:600}.flow-guide small{display:block;color:var(--muted);font-size:11px}.bypass .channel{stroke-dasharray:6 4}.bypass .flow{opacity:.45}.bank{stroke:#c4a877;stroke-width:1;fill:#756447;fill-opacity:.15}.basin{fill:url(#reservoir-fill);stroke:#83c5c2;stroke-width:1.4;transition:opacity .4s}.reservoir-label{fill:#dcf2e6;font:600 13px 'Segoe UI',sans-serif;letter-spacing:1px}.reservoir-sub{fill:#afd1c7;font:11px 'Segoe UI',sans-serif}.feature[data-feature=jay] .reservoir-label{font-size:10px}.feature[data-feature=jay] .reservoir-sub{font-size:8px}.moat{fill:#20505a;stroke:#72b6b6;stroke-width:1.4}.enclosure{fill:#182a2d;stroke:#d5b16e;stroke-width:1.3}.temple-inner,.temple-lines{fill:none;stroke:#cdb076;stroke-width:1}.tower{fill:#e2bf81}.temple-label{fill:#e5c68f;font:11px 'Segoe UI',sans-serif;letter-spacing:1px;paint-order:stroke;stroke:var(--bg);stroke-width:5px;stroke-linejoin:round}.lake-label{fill:#7fc6c7;font:24px Georgia,serif;letter-spacing:6px}.pond{fill:#366f70;stroke:#8fb8ab;stroke-width:.7}.feature{cursor:pointer;outline:none}.hit{fill:#152b2e;fill-opacity:.65;stroke:transparent}.selection-ring{fill:none;stroke:transparent;stroke-width:2;stroke-dasharray:3 3}.feature:hover .selection-ring,.feature.selected .selection-ring,.feature:focus-visible .selection-ring{stroke:#f2d799}.feature:hover .hit,.feature.selected .hit,.feature:focus-visible .hit{stroke:#d6ba79}.feature:hover .fields,.feature.selected .fields,.feature:focus-visible .fields{stroke:#d6ba79;stroke-width:2}.feature:hover .lake,.feature.selected .lake,.feature:focus-visible .lake{stroke:#d6ba79;stroke-opacity:1}.feature:focus-visible{filter:drop-shadow(0 0 4px #b9e0d6)}[data-zone]{transition:opacity .3s}.dim{opacity:.18}.area-box{fill:#245157;stroke:#83c5c2;stroke-width:1}.area-value{fill:#dae8d7;font:26px Georgia,serif}.paused .flow{animation-play-state:paused}.dry .inflow .flow,.dry .bypass .flow{opacity:.12}.dry .inflow .channel,.dry .bypass .channel{opacity:.35}.dry .basin{opacity:.65}.dry .flow{animation-duration:5s}.dry .fields{opacity:.55}@keyframes flow{to{stroke-dashoffset:-81}}@media(prefers-reduced-motion:reduce){.flow{animation:none!important}*{transition:none!important}}
@media(min-width:1500px){main{padding-left:30px;padding-right:30px}}@media(max-width:1100px){.hero{grid-template-columns:1fr 330px}.atlas{grid-template-columns:minmax(0,1fr) 250px}.inspector{padding:20px 17px}.inspector h2{font-size:27px}.map-top{font-size:10px}.secondary{gap:30px}}@media(max-width:800px){main{padding:20px 20px 35px}.hero{grid-template-columns:1fr;gap:22px;padding-top:10px}.intro{max-width:60ch}.hero-meta{max-width:450px}.toolbar{flex-wrap:wrap}.atlas{grid-template-columns:1fr}.inspector{border-left:0;border-top:1px solid var(--line);padding:22px}.inspector select{margin-bottom:17px}.season-note{margin-top:10px}.journey{grid-template-columns:1fr 1fr}.journey button:nth-child(2){border-right:0}.journey button:nth-child(-n+2){border-bottom:1px solid var(--line)}.story{grid-template-columns:1fr;gap:12px;margin-bottom:28px}.secondary,.evidence-grid{grid-template-columns:1fr;gap:30px}.map-hint:after{content:' On small screens, swipe the drawing sideways.'}.tool-label{display:none}h1{font-size:80px}.map-top{padding:12px}.legend{gap:12px;padding:12px}.journey span{font-size:17px}.journey button{padding:12px}.site-brand{margin-left:20px!important}}@media print{.toolbar,.journey,.inspector select,.inspector label{display:none}.atlas{display:block}.inspector{border:0}body{print-color-adjust:exact}.flow{animation:none}.dim{opacity:1}.evidence{display:block}}
"""

In [12]:
script = r"""
const model = __MODEL__;
const $ = (id) => document.getElementById(id);
const motion = window.matchMedia('(prefers-reduced-motion: reduce)');
const state = {season:'wet', stage:0, feature:'west', paused:motion.matches, zoom:false};
// Place a fixed arrow along the actual SVG path. Both this arrow and the
// animated dashes follow its start-to-end order; neither reverses by season.
for (const route of document.querySelectorAll('#network .route')) {
  const path = route.querySelector('.direction');
  const distance = path.getTotalLength() / 2;
  const point = path.getPointAtLength(distance);
  const before = path.getPointAtLength(Math.max(0, distance - 1));
  const after = path.getPointAtLength(distance + 1);
  const angle = Math.atan2(after.y - before.y, after.x - before.x) * 180 / Math.PI;
  const arrow = document.createElementNS('http://www.w3.org/2000/svg', 'path');
  arrow.setAttribute('class', 'flow-arrow');
  arrow.setAttribute('d', 'M-5 -4L0 0L-5 4');
  arrow.setAttribute('transform', `translate(${point.x} ${point.y}) rotate(${angle})`);
  arrow.setAttribute('aria-hidden', 'true');
  route.append(arrow);
}
function render() {
  document.body.classList.toggle('dry',state.season==='dry');
  document.body.classList.toggle('paused',state.paused || document.hidden || motion.matches);
  $('motion').textContent = motion.matches ? 'Motion reduced' : state.paused ? 'Play flow' : 'Pause flow';
  $('motion').disabled = motion.matches;
  $('motion').setAttribute('aria-pressed',String(state.paused || motion.matches));
  for(const button of document.querySelectorAll('[data-season]')) button.setAttribute('aria-pressed',String(button.dataset.season===state.season));
  for(const button of document.querySelectorAll('[data-step]')) button.setAttribute('aria-pressed',String(Number(button.dataset.step)===state.stage));
  for(const node of document.querySelectorAll('#network [data-zone]')) node.classList.toggle('dim',state.stage!==0 && Number(node.dataset.zone)!==state.stage);
  for(const node of document.querySelectorAll('#network [data-feature]')) {
    const selected=node.dataset.feature===state.feature;
    node.classList.toggle('selected',selected);
    node.setAttribute('aria-pressed',String(selected));
  }
  const f=model.features[state.feature], season=model.seasons[state.season], stage=model.stages[state.stage];
  $('feature-select').value=state.feature;
  $('inspect-kind').textContent=f.kind;
  $('inspect-name').textContent=f.name;
  $('inspect-metric').textContent=f.metric;
  $('inspect-body').textContent=f.body;
  $('inspect-detail').textContent=f.detail;
  $('inspect-source').textContent=model.sources[f.source].label+' ↗';
  $('inspect-source').href=model.sources[f.source].url;
  $('season-heading').textContent=season.heading;
  $('season-text').textContent=season.text;
  $('season-label').textContent=season.label+' / qualitative view';
  $('story-heading').textContent=stage.heading;
  $('story-text').textContent=stage.text;
  $('network').classList.toggle('zoomed',state.zoom);
  $('zoom').setAttribute('aria-pressed',String(state.zoom));
  $('zoom').textContent=state.zoom?'Fit diagram':'Enlarge diagram';
}
function chooseFeature(key) {
  state.feature=key;
  // Selecting any component restores the full network context.
  state.stage=0;
  render();
}
for(const node of document.querySelectorAll('#network [data-feature]')) {
  node.addEventListener('click',()=>chooseFeature(node.dataset.feature));
  node.addEventListener('keydown',(event)=>{
    if(event.key==='Enter'||event.key===' ') {event.preventDefault();chooseFeature(node.dataset.feature);}
  });
}
$('feature-select').addEventListener('change',event=>chooseFeature(event.target.value));
for(const button of document.querySelectorAll('[data-season]')) button.addEventListener('click',()=>{state.season=button.dataset.season;render();});
for(const button of document.querySelectorAll('[data-step]')) button.addEventListener('click',()=>{
  state.stage=Number(button.dataset.step);
  if(model.stages[state.stage].feature) state.feature=model.stages[state.stage].feature;
  render();
});
$('motion').addEventListener('click',()=>{state.paused=!state.paused;render();});
$('zoom').addEventListener('click',()=>{state.zoom=!state.zoom;render();});
$('reset').addEventListener('click',()=>{
  Object.assign(state,{season:'wet',stage:0,feature:'west',paused:motion.matches,zoom:false});
  $('map-scroll').scrollLeft=0;
  render();
});
document.addEventListener('visibilitychange',render);
motion.addEventListener('change',()=>{state.paused=motion.matches;render();});
render();
""".replace('__MODEL__', json.dumps(model, ensure_ascii=False).replace('</', '<\\/'))

## Export

`write_svg_page` supplies the same branding, local stylesheet and legal footer used by the other notebook exports, without loading Plotly. The HTML needs only those shared site assets and is ready to serve from `output/`. The static SVG and initial West Baray explanation remain visible without JavaScript.

In [13]:
options = ''.join(f'<option value="{k}"{ " selected" if k == "west" else ""}>{esc(v["name"])}</option>' for k,v in features.items())
stage_buttons = ''.join(f'<button type="button" data-step="{i}" aria-pressed="{str(i==0).lower()}"><small>{"OVERVIEW" if i==0 else f"0{i} / THE JOURNEY"}</small><span>{esc(s["name"])}</span></button>' for i,s in enumerate(stages))
initial = features['west']
template = r"""<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<meta name="visualization-engine" content="svg">
<meta name="description" content="An interactive engineering atlas of Angkor’s historical water system: catch the monsoon, store water, distribute it and drain the surplus.">
<title>Angkor · The city that moved water | phey.app</title><style>__CSS__</style></head>
<body><main>
<header class="hero"><div><p class="eyebrow">An atlas of ingenuity / Cambodia</p><h1>ANGKOR<span>The city that moved water.</span></h1></div>
<div><p class="intro">Behind the temples lay a landscape engineered for water. Follow the monsoon from the northern hills to the reservoirs, fields and lake.</p><div class="hero-meta"><div><strong>&gt;1,000 km²</strong><span>NETWORK EXTENT <a href="https://pubmed.ncbi.nlm.nih.gov/18926615/" aria-label="Source for network extent">↗</a></span></div><div><strong>9th–13th c.</strong><span>A LANDSCAPE IN THE MAKING</span></div></div></div></header>
<div class="toolbar"><div class="tool-group" role="group" aria-label="Season"><span class="tool-label">EXPLORE A SEASON</span><button type="button" data-season="wet" aria-pressed="true">Monsoon</button><button type="button" data-season="dry" aria-pressed="false">Dry season</button></div><div class="tool-group"><button type="button" class="small-button" id="motion" aria-pressed="false">Pause flow</button><button type="button" class="small-button" id="reset">Reset view</button></div></div>
<section class="atlas" aria-label="Interactive historical water atlas"><div class="map-panel"><div class="map-top"><span><i class="live-dot"></i><span id="season-label">Monsoon / qualitative view</span></span><button type="button" class="small-button" id="zoom" aria-pressed="false">Enlarge diagram</button></div><p class="flow-guide"><strong>Regional flow:</strong> Northern runoff → reservoirs → southern drainage.<small>Surplus flows toward Tonle Sap. Canal routes are schematic. <a href="https://doi.org/10.1017/S0003598X00097295">Evidence ↗</a></small></p><div class="map-scroll" id="map-scroll" tabindex="0" role="region" aria-label="Scrollable water system diagram">__NETWORK__</div><p class="map-hint">Select a named feature to explore its role.</p><div class="legend"><span><i class="water-key"></i>Water network</span><span><i class="gold-key"></i>Enclosures & banks</span><span><i class="green-key"></i>Agricultural plain</span><span><i class="arrow-key">→</i>Downstream direction</span></div></div>
<aside class="inspector" aria-label="Feature details"><label for="feature-select">EXPLORE A COMPONENT</label><select id="feature-select">__OPTIONS__</select><div aria-live="polite" aria-atomic="true"><p class="inspect-kind" id="inspect-kind">02 / STORE</p><h2 id="inspect-name">West Baray</h2><p class="metric" id="inspect-metric">8 × 2 km</p><p class="inspect-body" id="inspect-body">__INITIAL_BODY__</p><p class="inspect-detail" id="inspect-detail">__INITIAL_DETAIL__</p><a class="source-link" id="inspect-source" href="https://doi.org/10.1017/S0003598X00097295">Fletcher et al. · 2008 ↗</a></div><div class="season-note" aria-live="polite"><h3 id="season-heading">When water is abundant</h3><p id="season-text">__SEASON_TEXT__</p></div></aside></section>
<nav class="journey" aria-label="Follow the water">__STAGES__</nav><div class="story" aria-live="polite"><h2 id="story-heading">An engineered landscape</h2><p id="story-text">__STORY_TEXT__</p></div>
<section class="secondary"><article><p class="eyebrow">01 / The scale of storage</p><h2>Water at a monumental scale.</h2><p>Two great reservoirs, drawn at the same scale. Their long, straight banks turn an abstract area into something you can see.</p>__COMPARISON__<p class="caption">Approximate rectangular footprints, calculated from published dimensions. Area does not indicate depth, capacity or how much water was present. <a href="https://doi.org/10.1017/S0003598X00097295">West Baray</a> · <a href="https://www.iccrom.org/sites/default/files/publications/2020-10/traditional-knowledge-systems.pdf">East Baray</a></p></article>
<article><p class="eyebrow">02 / The engineering principle</p><h2>Work with the fall of the land.</h2><p>Read the landscape in section: collection above, retention in the middle, distribution and drainage below. This is an explanatory profile, with the slope exaggerated.</p>
<svg viewBox="0 0 590 235" role="img" aria-labelledby="profile-title"><title id="profile-title">Conceptual north to south section: uplands, retention and southern drainage. No measured elevation scale.</title><defs><marker id="section-arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="5" markerHeight="5" orient="auto"><path d="M1 1L8 5L1 9" fill="none" stroke="#7bd6d0" stroke-width="1.3"/></marker></defs><path class="section-ground" d="M15 76L70 50L115 96L160 114L210 130L242 130L249 115L258 155H330L338 129L347 151L402 164L465 183L573 190V215H15Z"/><path class="section-water" d="M26 68L72 43L118 89L166 108L226 124" marker-end="url(#section-arrow)"/><path d="M259 137H331V154H259Z" fill="#397980"/><path class="section-water" d="M345 145L405 158L473 177L548 183" marker-end="url(#section-arrow)"/><path d="M490 185H574V195H490Z" fill="#397980"/><path d="M294 89V128M393 104V151" stroke="#8da29b" stroke-dasharray="2 3"/><text class="section-label" x="15" y="25">N / Uplands</text><text class="section-label" x="253" y="73">Retain</text><text class="section-label" x="358" y="90">Distribute</text><text class="section-label" x="489" y="163">S / Lake</text><text class="small-label" x="15" y="233">CONCEPTUAL SECTION · VERTICAL SCALE EXAGGERATED</text></svg>
<p class="caption">The ingenuity lies in controlling where water moves—and where it stays. Canals, banks, moats and reservoirs form a connected landscape. <a href="https://doi.org/10.1017/S0003598X00097295">Three-zone interpretation ↗</a></p></article></section>
<details class="evidence"><summary>About this reconstruction · evidence, uncertainty & sources</summary><div class="evidence-grid"><div><h3>A system assembled over centuries.</h3><p>This is a cumulative schematic of ninth–thirteenth-century infrastructure, not one moment of “peak operation.” The main diagram simplifies relative placement. Canal connections, field parcels, contours and shoreline are illustrative; only the reservoir comparison has a metric scale.</p><p>The seasonal views and moving dashes explain roles and direction. They do not encode measured flow, historical water levels or crop yields. Reservoirs need not have operated simultaneously. Ritual, domestic and agricultural uses cannot be reduced to a single function.</p><h3>Powerful, yet dependent on maintenance.</h3><p>Research models suggest that extreme flooding could destabilize the network through erosion and sedimentation. This is one contribution to understanding Angkor’s transformation, not a single-cause explanation of its decline. <a href="https://pmc.ncbi.nlm.nih.gov/articles/PMC6192684/">Penny et al. (2018)</a>.</p></div><div><h3>Read the evidence.</h3><ul><li><a href="https://doi.org/10.1017/S0003598X00097295">Fletcher et al. (2008)</a> · Archaeological network, functional zones and West Baray dimensions.</li><li><a href="https://pubmed.ncbi.nlm.nih.gov/18926615/">Kummu (2009)</a> · Network extent, seasonality and altered hydrology.</li><li><a href="https://pmc.ncbi.nlm.nih.gov/articles/PMC6192684/">Penny et al. (2018)</a> · Network development, Jayatataka and vulnerability to climatic variation.</li><li><a href="https://www.iccrom.org/sites/default/files/publications/2020-10/traditional-knowledge-systems.pdf">ICCROM (2020), chapter 6, p. 90</a> · East Baray chronology and dimensions.</li></ul><p>Original vector diagrams. All geometry, data and interaction code are documented in the <a href="../notebooks/20261031_angkor_hydraulic_city.ipynb">Jupyter notebook ↗</a>. Regenerates offline; no map service or external fonts.</p></div></div></details>
<p class="page-note">Historical engineering atlas · Interpretive geometry, sourced dimensions · phey.app</p>
<noscript><p>JavaScript is disabled. The diagram, reservoir comparison and sources remain readable; enable JavaScript to use the controls.</p></noscript>
</main><script>__SCRIPT__</script></body></html>"""
replacements = {'CSS':css, 'NETWORK':network_svg, 'OPTIONS':options, 'STAGES':stage_buttons,
                'COMPARISON':comparison_svg, 'SCRIPT':script,
                'INITIAL_BODY':esc(initial['body']), 'INITIAL_DETAIL':esc(initial['detail']),
                'SEASON_TEXT':esc(seasons['wet']['text']), 'STORY_TEXT':esc(stages[0]['text'])}
document = template
for key,value in replacements.items():
    document = document.replace('__'+key+'__',value)
assert not any('__'+key+'__' in document for key in replacements)
assert document.count('<h1>') == 1
output_file.parent.mkdir(exist_ok=True)
write_svg_page(output_file, document)
print(f'Exported {output_file.relative_to(project_root)} ({output_file.stat().st_size:,} bytes)')

Exported output\20261031_angkor_hydraulic_city.html (47,811 bytes)


In [14]:
# Render the actual exported page rather than a second notebook-only plot.
relative_export = '../output/' + output_file.name if current.name == 'notebooks' else 'output/' + output_file.name
display(IFrame(src=relative_export, width='100%', height=1000))